# Tropical Cyclone Intensity Classification from Satellite Imagery using Deep Learning

**CIA 3 Project — Computer Vision (Deep Learning-based approach)**

Pipeline: INSAT3D infrared satellite images → transfer-learning CNN → intensity category classification → Grad-CAM explainability → evaluation.

**Confirmed dataset structure** (from inspection): all 418 images sit in a single flat folder `CYCLONE_DATASET_INFRARED/`, and each filename IS the wind speed in knots directly (e.g. `50.jpg` = 50 knots). A `(n)` suffix like `45(2).jpg` just disambiguates duplicate images at the same knot value.

**Run this in Google Colab:**
1. Runtime → Change runtime type → T4 GPU → Save
2. Get `kaggle.json` from https://www.kaggle.com/settings → API → Create New Token
3. Run cells in order — first cell prompts a `kaggle.json` upload

## 1. Setup & Dataset Download

In [ ]:
!pip install kaggle grad-cam -q

import os
from google.colab import files

print('Upload your kaggle.json now:')
uploaded = files.upload()
os.makedirs('/root/.kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

In [ ]:
!kaggle datasets download -d sshubam/insat3d-infrared-raw-cyclone-images-20132021 -p /content/data --unzip

## 2. Imports

In [ ]:
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report, f1_score
from sklearn.model_selection import train_test_split
import seaborn as sns

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)
torch.manual_seed(42)
np.random.seed(42)

## 3. Build Labeled Dataframe — Using Official CSV Labels

The dataset includes an official label file (`insat_3d_ds - Sheet.csv`) mapping each `img_name` to its `label` (wind speed in knots) — confirmed to match the filename-encoded values exactly on inspection. We use this CSV directly as the authoritative label source.

Bucketed into the **IMD 5-class intensity scale**:

| Category | Wind Speed (knots) |
|---|---|
| Depression | < 34 |
| Cyclonic Storm (CS) | 34–47 |
| Severe CS | 48–63 |
| Very Severe CS | 64–89 |
| Extremely Severe CS | 90+ |

Labels are sourced from the dataset's own official label file (`insat_3d_ds - Sheet.csv`, columns `img_name` and `label`), which we've confirmed matches the filename-encoded knot values exactly. Using the official CSV is more robust than parsing filenames and is the correct approach for the final pipeline.

Bucketed into the **IMD 5-class intensity scale**:

| Category | Wind Speed (knots) |
|---|---|
| Depression | < 34 |
| Cyclonic Storm (CS) | 34–47 |
| Severe CS | 48–63 |
| Very Severe CS | 64–89 |
| Extremely Severe CS | 90+ |

In [ ]:
DATA_DIR = '/content/data/insat3d_ir_cyclone_ds/CYCLONE_DATASET_INFRARED'
LABELS_CSV = '/content/data/insat_3d_ds - Sheet.csv'

def knots_to_category(knots):
    if knots < 34:
        return 'Depression'
    elif knots < 48:
        return 'Cyclonic Storm'
    elif knots < 64:
        return 'Severe CS'
    elif knots < 90:
        return 'Very Severe CS'
    else:
        return 'Extremely Severe CS'

labels_df = pd.read_csv(LABELS_CSV)
print('CSV rows:', len(labels_df))

records = []
for _, row in labels_df.iterrows():
    img_path = os.path.join(DATA_DIR, row['img_name'])
    if os.path.exists(img_path):
        knots = int(row['label'])
        records.append({'path': img_path, 'knots': knots, 'category': knots_to_category(knots)})

df = pd.DataFrame(records)
print('\nMatched to real image files:', len(df), 'of', len(labels_df), 'CSV rows')
print('\nCategory distribution:')
print(df['category'].value_counts())
df.head()

**Check the output above before continuing.** If "Matched to real image files" is less than the CSV row count, some filenames in the CSV don't exist in the folder — worth investigating before continuing. If the category distribution looks heavily skewed (expected — extreme storms are naturally rarer) that's fine, we handle it with class weighting below.

## 4. Train / Val / Test Split

With only ~418 images total, splits will be small — this is a real limitation worth noting in your report. We stratify to preserve class balance as much as possible.

In [ ]:
CLASSES = ['Depression', 'Cyclonic Storm', 'Severe CS', 'Very Severe CS', 'Extremely Severe CS']
class_to_idx = {c: i for i, c in enumerate(CLASSES)}
df['label'] = df['category'].map(class_to_idx)

# Drop any classes with fewer than 2 samples (can't stratify-split those)
class_counts_check = df['label'].value_counts()
valid_classes = class_counts_check[class_counts_check >= 2].index
df = df[df['label'].isin(valid_classes)].reset_index(drop=True)

train_df, temp_df = train_test_split(df, test_size=0.3, stratify=df['label'], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df['label'], random_state=42)

print('Train:', len(train_df), '| Val:', len(val_df), '| Test:', len(test_df))

## 5. Dataset & DataLoaders

Heavier augmentation than usual, since the dataset is small — this helps the model generalize from limited examples.

In [ ]:
IMG_SIZE = 224

train_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

eval_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

class CycloneDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row['path']).convert('RGB')
        img = self.transform(img)
        return img, row['label']

BATCH_SIZE = 16
train_loader = DataLoader(CycloneDataset(train_df, train_tfms), batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(CycloneDataset(val_df, eval_tfms), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(CycloneDataset(test_df, eval_tfms), batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

## 6. Model — Transfer Learning (ResNet18)

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, len(CLASSES))
model = model.to(device)

class_counts = train_df['label'].value_counts().reindex(range(len(CLASSES)), fill_value=1).values
class_weights = torch.tensor(1.0 / class_counts, dtype=torch.float32)
class_weights = (class_weights / class_weights.sum() * len(CLASSES)).to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(model.parameters(), lr=1e-4)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)

## 7. Training Loop

In [ ]:
def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.set_grad_enabled(train):
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            if train:
                optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, labels)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * imgs.size(0)
            preds = outputs.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return total_loss / total, correct / total

EPOCHS = 20
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_acc = 0

for epoch in range(EPOCHS):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)
    scheduler.step(val_loss)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    print(f'Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} Acc: {train_acc:.4f} | '
          f'Val Loss: {val_loss:.4f} Acc: {val_acc:.4f}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')
        print('  -> Saved new best model')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], label='Train'); axes[0].plot(history['val_loss'], label='Val')
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].plot(history['train_acc'], label='Train'); axes[1].plot(history['val_acc'], label='Val')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()
plt.tight_layout(); plt.savefig('training_curves.png', dpi=150); plt.show()

## 8. Evaluation on Test Set

In [ ]:
model.load_state_dict(torch.load('best_model.pth'))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        outputs = model(imgs)
        preds = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

present_labels = sorted(set(all_labels) | set(all_preds))
present_names = [CLASSES[i] for i in present_labels]
print(classification_report(all_labels, all_preds, labels=present_labels, target_names=present_names))
print('Macro F1:', f1_score(all_labels, all_preds, average='macro'))

cm = confusion_matrix(all_labels, all_preds, labels=present_labels)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=present_names, yticklabels=present_names, cmap='Blues')
plt.xlabel('Predicted'); plt.ylabel('True'); plt.title('Confusion Matrix')
plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.savefig('confusion_matrix.png', dpi=150); plt.show()

## 9. Grad-CAM Explainability

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

target_layer = [model.layer4[-1]]
cam = GradCAM(model=model, target_layers=target_layer)

def visualize_gradcam(sample_row):
    img = Image.open(sample_row['path']).convert('RGB')
    input_tensor = eval_tfms(img).unsqueeze(0).to(device)

    outputs = model(input_tensor)
    probs = torch.softmax(outputs, dim=1)[0]
    pred_idx = outputs.argmax(dim=1).item()
    confidence = probs[pred_idx].item()

    grayscale_cam = cam(input_tensor=input_tensor, targets=[ClassifierOutputTarget(pred_idx)])[0]
    rgb_img = np.array(img.resize((IMG_SIZE, IMG_SIZE))) / 255.0

    # IR images are already rainbow/jet-colored, same scheme Grad-CAM uses --
    # overlaying jet-on-jet hides the heatmap. Use grayscale base for contrast.
    gray_img = np.array(img.convert('L').resize((IMG_SIZE, IMG_SIZE))) / 255.0
    gray_img_3ch = np.stack([gray_img] * 3, axis=-1)
    visualization = show_cam_on_image(gray_img_3ch, grayscale_cam, use_rgb=True)

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))
    axes[0].imshow(rgb_img); axes[0].set_title(f'Original (true: {sample_row["category"]}, {sample_row["knots"]}kt)'); axes[0].axis('off')
    axes[1].imshow(visualization)
    axes[1].set_title(f'Grad-CAM: {CLASSES[pred_idx]} ({confidence*100:.1f}%)')
    axes[1].axis('off')
    plt.tight_layout()
    plt.savefig(f'gradcam_{sample_row.name}.png', dpi=150)
    plt.show()

for cat in test_df['category'].unique():
    subset = test_df[test_df['category'] == cat]
    if len(subset) > 0:
        print(f'--- True category: {cat} ---')
        visualize_gradcam(subset.iloc[0])

## 10. Summary for Report

- Total dataset size: 418 images (note this as a limitation — small dataset for deep learning, mitigated via transfer learning + heavy augmentation)
- Best validation accuracy (from training log)
- Test set classification report (per-class precision/recall/F1)
- Macro F1 score
- Confusion matrix (`confusion_matrix.png`)
- Training curves (`training_curves.png`)
- Grad-CAM visualizations per class (`gradcam_*.png`)